# Full-run test for ADASYN

This notebook provides runnable cells to perform a full training run for 'ADASYN' using the project `TrainTestSplitPipeline`.

Notes:
- ADASYN is adaptive and generates more samples for hard-to-learn minority examples
- Similar to SMOTE but with density-based weighting

In [ ]:
# Colab setup — run this cell first, then restart and run all
# Skip if running locally (cell will no-op outside Colab)
import subprocess, sys, os
from pathlib import Path

def _running_on_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _running_on_colab():
    REPO_PATH = '/content/Katabatic'
    if not os.path.exists(REPO_PATH):
        subprocess.run(
            ['git', 'clone', 'https://github.com/lukebrumby/katabatic-personal.git', REPO_PATH],
            check=True
        )
    os.chdir(REPO_PATH)
    sys.path.insert(0, REPO_PATH)
    print(f'Colab: repo at {REPO_PATH}, CWD set to {os.getcwd()}')
else:
    # Local: walk up to repo root (finds pyproject.toml)
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'pyproject.toml').exists():
            break
        ROOT = ROOT.parent
    os.chdir(ROOT)
    sys.path.insert(0, str(ROOT))
    print(f'Local: CWD set to {ROOT}')

In [1]:
# Install dependencies
%pip install imbalanced-learn xgboost

In [2]:
# Imports and helpers
import os
import importlib
import traceback
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

def ensure(path):
    os.makedirs(path, exist_ok=True)

def make_pipeline(model_callable, skip_evaluations=True, evaluations=None):
    if skip_evaluations:
        return TrainTestSplitPipeline(model=model_callable, evaluations=[], override_evaluations=True)
    elif evaluations is not None:
        return TrainTestSplitPipeline(model=model_callable, evaluations=evaluations, override_evaluations=True)
    else:
        return TrainTestSplitPipeline(model=model_callable)

ModuleNotFoundError: No module named 'katabatic'

In [ ]:
# Configuration
DATASETS = ['adult', 'car', 'magic', 'nursery', 'shuttle']
SKIP_EVALUATIONS = False  # Set to True to skip TSTR evaluations

# Top-level dirs
ensure('discretized_data')
ensure('sample_data')
ensure('synthetic')
ensure('Results')

MODEL_MAP = {
    'adasyn': ('katabatic.models.adasyn.models', 'ADASYNModel'),
}

ADASYN_CONFIG = {
    'n_neighbors': 5,
    'sampling_strategy': 'auto',
    'random_state': 42,
}

## Preprocess datasets (run once)

In [ ]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'Preprocessing {dataset}...')
    try:
        discretize_preprocess(
            file_path=f'raw_data/{dataset}.csv', 
            output_path=f'discretized_data/{dataset}.csv', 
            bins=10, 
            strategy='uniform'
        )
        print(f'Discretized -> discretized_data/{dataset}.csv')
    except Exception as e:
        print(f'Failed to preprocess {dataset}: {e}')
        traceback.print_exc()

## Run ADASYN

In [ ]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'ADASYN -> {dataset}')
    synth_dir = os.path.join('synthetic', dataset, 'adasyn')
    ensure(synth_dir)
    try:
        mod_path, cls_name = MODEL_MAP['adasyn']
        module = importlib.import_module(mod_path)
        ModelClass = getattr(module, cls_name)
        model_factory = lambda: ModelClass(**ADASYN_CONFIG)
        pipeline = make_pipeline(model_factory, skip_evaluations=SKIP_EVALUATIONS)
        result = pipeline.run(
            input_csv=f'discretized_data/{dataset}.csv',
            output_dir=f'sample_data/{dataset}',
            synthetic_dir=synth_dir,
            real_test_dir=f'sample_data/{dataset}'
        )
        print('ADASYN finished for', dataset)
    except Exception as e:
        print('ADASYN failed for', dataset, e)
        traceback.print_exc()